# HW3 - Q3 [35 pts]

## Important Notices

<div class="alert alert-block alert-danger">
    WARNING: <strong>REMOVE</strong> any print statements added to cells with "#export" that are used for debugging purposes befrore submitting because they will crash the autograder in Gradescope. Any additional cells can be used for testing purposes at the bottom. 
</div>

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> import any additional libraries into this workbook.
</div>

All instructions, code comments, etc. in this notebook **are part of the assignment instructions**. That is, if there is instructions about completing a task in this notebook, that task is not optional.  

<div class="alert alert-block alert-info">
    You <strong>must</strong> implement the following functions in this notebook to receive credit.
</div>

`user()`

`long_trips()`

`manhattan_trips()`

`weighted_profit()`

`final_output()`

Each method will be auto-graded using different sets of parameters or data, to ensure that values are not hard-coded.  You may assume we will only use your code to work with data from the NYC-TLC dataset during auto-grading.

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove or modify the following utility functions:
</div>

`load_data()`

`main()`

<div class="alert alert-block alert-info">
    Do <strong>not</strong> change the below cell. Run it to initialize your PySpark instance. If you don't get any output, make sure your Notebook's Kernel is set to "PySpark" in the top right corner.
</div>

In [1]:
sc

''

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remodify the below cell. It contains the function for loading data and all imports, and the function for running your code.
</div>

In [3]:
#export
from pyspark.sql.functions import *
from pyspark.sql import *

In [1]:
#### DO NOT CHANGE ANYTHING IN THIS CELL ####

def load_data(size='small'):
    # Loads the data for this question. Do not change this function.
    # This function should only be called with the parameter 'small' or 'large'
    
    if size != 'small' and size != 'large':
        print("Invalid size parameter provided. Use only 'small' or 'large'.")
        return
    
    input_bucket = "s3://cse6242-hw3-q3"
    
    # Load Trip Data
    trip_path = '/'+size+'/yellow_tripdata*'
    trips = spark.read.csv(input_bucket + trip_path, header=True, inferSchema=True)
    print("Trip Count: ",trips.count()) # Prints # of trips (# of records, as each record is one trip)
    
    # Load Lookup Data
    lookup_path = '/'+size+'/taxi*'
    lookup = spark.read.csv(input_bucket + lookup_path, header=True, inferSchema=True)
    
    return trips, lookup

def main(size, bucket):
    # Runs your functions implemented above.
    
    print(user())
    trips, lookup = load_data(size=size)
    trips = long_trips(trips)
    mtrips = manhattan_trips(trips, lookup)
    wp = weighted_profit(trips, mtrips)
    final = final_output(wp, lookup)
    
    # Outputs the results for you to visually see
    final.show()
    
    # Writes out as a CSV to your bucket.
    final.write.csv(bucket)

# Implement the below functions for this assignment:
<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> change any function inputs or outputs, and ensure that the dataframes your code returns align with the schema definitions commented in each function. Do <strong>NOT</strong> remove the #export comment from each of the code blocks either. This can prevent your code from being converted to a python file.
</div>

## 3.1 [1 pt] Update the `user()` function
This function should return your GT username, eg: gburdell3

In [3]:
#export
def user():
    # Returns a string consisting of your GT username.
    return 'nshukla46'

# Call the function and print the output
print(user())

nshukla46


## 3.2 [2 pts] Update the `long_trips()` function
This function filters trips to keep only trips greater than or equal to 2 miles.

In [5]:
#export
def long_trips(trips):
    """
    Filters the trips DataFrame to keep only trips greater than or equal to 2 miles.
    
    Parameters:
        trips (pd.DataFrame): DataFrame containing trip data with a 'distance' column.
    
    Returns:
        pd.DataFrame: Filtered DataFrame containing trips with a distance of at least 2 miles.
    """
    import pandas as pd
    
    # Filter trips where 'distance' is greater than or equal to 2
    filtered_trips = trips[trips['distance'] >= 2]
    return filtered_trips


In [7]:
import pandas as pd

# Example data for testing
trips_data = {
    'trip_id': [1, 2, 3, 4, 5],
    'distance': [1.5, 2.0, 3.1, 1.9, 2.5],
    'passenger_count': [1, 2, 1, 3, 2]
}
trips = pd.DataFrame(trips_data)

# Call the function
result = long_trips(trips)
print(result)

   trip_id  distance  passenger_count
1        2       2.0                2
2        3       3.1                1
4        5       2.5                2


## 3.3 [6 pts] Update the `manhattan_trips()` function

This function determines the top 20 locations with a `DOLocationID` in manhattan by passenger_count (pcount).

Example output formatting:

```
+--------------+--------+
| DOLocationID | pcount |
+--------------+--------+
|             5|      15|
|            16|      12| 
+--------------+--------+
```

In [9]:
def manhattan_trips_top_20(trips, lookup):
    """
    Determines the top 20 locations in Manhattan by passenger count.
    
    Parameters:
        trips (pd.DataFrame): DataFrame containing trip data with columns 'DOLocationID' and 'passenger_count'.
        lookup (pd.DataFrame): DataFrame containing location lookup data with columns 'LocationID' and 'Borough'.
    
    Returns:
        list: A list of tuples, where each tuple contains (DOLocationID, pcount) for the top 20 locations.
    """
    import pandas as pd
    
    # Filter the lookup DataFrame to include only locations in Manhattan
    manhattan_locations = lookup[lookup['Borough'] == 'Manhattan']['LocationID']
    
    # Filter the trips DataFrame to include only trips with a DOLocationID in Manhattan
    manhattan_trips = trips[trips['DOLocationID'].isin(manhattan_locations)]
    
    # Group by DOLocationID and sum the passenger counts
    top_locations = manhattan_trips.groupby('DOLocationID')['passenger_count'].sum().reset_index()
    
    # Rename the 'passenger_count' column to 'pcount'
    top_locations = top_locations.rename(columns={'passenger_count': 'pcount'})
    
    # Sort the DataFrame by 'pcount' in descending order and select the top 20 locations
    top_20_locations = top_locations.sort_values(by='pcount', ascending=False).head(20)
    
    # Convert the DataFrame to a list of tuples
    top_20_list = list(top_20_locations.itertuples(index=False, name=None))
    
    return top_20_list

# Example usage:
top_20_output = manhattan_trips_top_20(trips, lookup)
print("Top 20 Manhattan Locations by Passenger Count:")
for location in top_20_output:
    print(f"DOLocationID: {location[0]}, Passenger Count: {location[1]}")


NameError: name 'lookup' is not defined

In [31]:
import pandas as pd

# Sample data for the trips DataFrame
trips_data = {
    'DOLocationID': [5, 16, 5, 16, 3, 5, 8, 8, 16, 3],
    'passenger_count': [5, 7, 4, 3, 2, 6, 3, 4, 2, 1]
}
trips = pd.DataFrame(trips_data)

# Sample data for the lookup DataFrame
lookup_data = {
    'LocationID': [5, 16, 3, 8],
    'Borough': ['Manhattan', 'Manhattan', 'Brooklyn', 'Manhattan']
}
lookup = pd.DataFrame(lookup_data)
manhattan_trips = pd.DataFrame(lookup_data)


In [43]:
def manhattan_trips(trips, lookup):
    # Step 1: Filter the lookup DataFrame to include only Manhattan locations
    manhattan_locations = lookup[lookup['Borough'] == 'Manhattan']['LocationID']
    
    # Step 2: Filter the trips DataFrame to include only trips with DOLocationID in Manhattan
    filtered_trips = trips[trips['DOLocationID'].isin(manhattan_locations)]
    
    # Step 3: Group by DOLocationID and sum the passenger_count
    grouped_trips = filtered_trips.groupby('DOLocationID')['passenger_count'].sum().reset_index()
    
    # Step 4: Sort by passenger_count in descending order and get the top 20
    top_20_locations = grouped_trips.sort_values(by='passenger_count', ascending=False).head(20)
    
    # Step 5: Return the top 20 DataFrame
    return top_20_locations


In [47]:
# Print the top 20 locations DataFrame
print(top_20_locations.to_string(index=False))



 DOLocationID  passenger_count
            5               15
           16               12
            8                7


## 3.4 [6 pts] Update the `weighted_profit()` function
This function should determine the average `total_amount`, the total count of trips, and the total count of trips ending in the top 20 destinations and return the `weighted_profit` as discussed in the homework document.

Example output formatting:
```
+--------------+-------------------+
| PULocationID |  weighted_profit  |
+--------------+-------------------+
|            18| 33.784444421924436| 
|            12| 21.124577637149223| 
+--------------+-------------------+
```

In [79]:
from tabulate import tabulate



In [83]:
def weighted_profit(trips, top_20_locations):
    """
    Determines the weighted profit for each DOLocationID based on average total_amount,
    the total count of trips, and the total count of trips ending in the top 20 destinations.
    Returns a DataFrame with DOLocationID and weighted_profit.
    """
    # Step 1: Calculate the average total_amount per DOLocationID
    avg_amount = trips.groupby('DOLocationID')['total_amount'].mean().reset_index()
    avg_amount = avg_amount.rename(columns={'total_amount': 'avg_total_amount'})
    # Step 2: Calculate the total count of trips per DOLocationID
    total_trip_count = trips.groupby('DOLocationID').size().reset_index(name='total_trip_count')
    # Step 3: Calculate the total count of trips ending in the top 20 destinations
    top_20_trip_count = trips[trips['DOLocationID'].isin(top_20_locations['DOLocationID'])].groupby('DOLocationID').size().reset_index(name='top_20_trip_count')
    # Step 4: Merge the DataFrames to consolidate the data
    merged_data = avg_amount.merge(total_trip_count, on='DOLocationID', how='left')
    merged_data = merged_data.merge(top_20_trip_count, on='DOLocationID', how='left')
    # Fill NaN values with 0 for the top_20_trip_count
    merged_data['top_20_trip_count'] = merged_data['top_20_trip_count'].fillna(0)
    
    # Step 5: Calculate the weighted_profit
    merged_data['weighted_profit'] = (merged_data['avg_total_amount'] * merged_data['top_20_trip_count']) / merged_data['total_trip_count']
     # Step 6: Return the DataFrame with DOLocationID and weighted_profit
    result = merged_data[['DOLocationID', 'weighted_profit']]
    # Step 7: Print the result in the specified format
    print(tabulate(result, headers=["PULocationID", "weighted_profit"], tablefmt="pipe", showindex=False))
    
    return result
    print(result)


In [97]:
# Example data
trips_data = {
    'PULocationID': [18, 18, 12, 12, 18, 14, 14, 18, 12, 16],
    'DOLocationID': [5, 16, 5, 8, 16, 8, 16, 8, 5, 5],
    'total_amount': [10.50, 20.75, 15.25, 30.00, 25.50, 18.75, 12.00, 22.50, 14.75, 19.25]
}
trips = pd.DataFrame(trips_data)

mtrips_data = {
    'DOLocationID': [5, 16, 8]
}
mtrips = pd.DataFrame(mtrips_data)

# Call the function and display the result
result = weighted_profit(trips, mtrips)

|   PULocationID |   weighted_profit |
|---------------:|------------------:|
|              5 |           14.9375 |
|              8 |           23.75   |
|             16 |           19.4167 |


## 3.5 [5 pts] Update the `final_output()` function
This function will take the results of `weighted_profit`, links it to the `borough` and `zone` and returns the top 20 locations with the highest `weighted_profit`.

Example output formatting:
```
+------------+---------+-------------------+
|    Zone    | Borough |  weighted_profit  |
+----------------------+-------------------+
| JFK Airport|   Queens|  16.95897820117925|
|     Jamaica|   Queens| 14.879835188762488|
+------------+---------+-------------------+
```

In [113]:
def weighted_profit(trips, top_20_locations):
    """
    Determines the weighted profit for each PULocationID based on average total_amount,
    the total count of trips, and the total count of trips ending in the top 20 destinations.
    Returns a DataFrame with PULocationID and weighted_profit.
    """
    # Step 1: Calculate the average total_amount per PULocationID
    avg_amount = trips.groupby('PULocationID')['total_amount'].mean().reset_index()
    avg_amount = avg_amount.rename(columns={'total_amount': 'avg_total_amount'})
    
    # Step 2: Calculate the total count of trips per PULocationID
    total_trip_count = trips.groupby('PULocationID').size().reset_index(name='total_trip_count')
    
    # Step 3: Calculate the total count of trips ending in the top 20 destinations
    top_20_trip_count = trips[trips['DOLocationID'].isin(top_20_locations['DOLocationID'])].groupby('PULocationID').size().reset_index(name='top_20_trip_count')
    
    # Step 4: Merge the DataFrames to consolidate the data
    merged_data = avg_amount.merge(total_trip_count, on='PULocationID', how='left')
    merged_data = merged_data.merge(top_20_trip_count, on='PULocationID', how='left')
    
    # Fill NaN values with 0 for the top_20_trip_count
    merged_data['top_20_trip_count'] = merged_data['top_20_trip_count'].fillna(0)
    
    # Step 5: Calculate the weighted_profit
    merged_data['weighted_profit'] = (merged_data['avg_total_amount'] * merged_data['top_20_trip_count']) / merged_data['total_trip_count']
    
    # Step 6: Return only PULocationID and weighted_profit columns
    result = merged_data[['PULocationID', 'weighted_profit']]

    return result


In [115]:
def final_output(weighted_profit_result, lookup):
    """
    This function takes the results of weighted_profit, links it to the borough and zone,
    and returns the top 20 locations with the highest weighted_profit.
    """
    # Step 1: Merge weighted_profit_result with lookup to get Zone and Borough
    merged_data = weighted_profit_result.merge(lookup, left_on='PULocationID', right_on='LocationID', how='left')
    
    # Step 2: Sort by weighted_profit in descending order
    sorted_data = merged_data.sort_values(by='weighted_profit', ascending=False)
    
    # Step 3: Select the top 20 locations
    top_20 = sorted_data.head(20)
    
    # Step 4: Select only Zone, Borough, and weighted_profit columns in the required order
    final_result = top_20[['Zone', 'Borough', 'weighted_profit']]

    # Step 5: Print the final result in the specified format
    print(tabulate(final_result, headers=["Zone", "Borough", "weighted_profit"], tablefmt="pipe", showindex=False))
    
    return final_result

# Call weighted_profit function and get the final output
weighted_profit_result = weighted_profit(trips, mtrips)
final_result = final_output(weighted_profit_result, lookup)


| Zone            | Borough   |   weighted_profit |
|:----------------|:----------|------------------:|
| Upper East Side | Manhattan |           20      |
| Midtown         | Manhattan |           19.8125 |
| Jamaica         | Queens    |           19.25   |
| Harlem          | Manhattan |           15.375  |


#### Testing

<div class="alert alert-block alert-info">
    You may use the below cell for any additional testing you need to do, however any code implemented below will not be run or used when grading.
</div>